# 05 - Bronze: ACLED events and weekly aggregates

This notebook extracts ACLED political violence, demonstration, and strategic development events for the 21 project countries.

It uses ACLED OAuth authentication, with credentials read from Databricks secrets. Do not paste ACLED credentials into notebook cells.

Expected secrets:

- scope: `cemac-secrets`
- key: `acled-username`
- key: `acled-password`

Outputs:

- `bronze.acled_events_historical`: event-level ACLED rows, one row per event
- `bronze.acled_weekly_aggregated`: country/admin1/week/event-type aggregates for dashboard use

The extraction paginates with `page=1`, `page=2`, etc. until a page returns fewer rows than `LIMIT`, matching ACLED's documented pagination pattern.

In [ ]:
# Cell 1 - Configuration
import time
from datetime import date, datetime, timezone
from typing import Dict, List

import requests
from pyspark.sql import Row
from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

TOKEN_URL = "https://acleddata.com/oauth/token"
ACLED_READ_URL = "https://acleddata.com/api/acled/read"

SECRET_SCOPE = "cemac-secrets"
USERNAME_SECRET_KEY = "acled-username"
PASSWORD_SECRET_KEY = "acled-password"

START_DATE = "2010-01-01"
END_DATE = date.today().isoformat()
LIMIT = 5000
MAX_PAGES_PER_COUNTRY_YEAR = 50
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5
SLEEP_BETWEEN_CALLS_SECONDS = 0.25

# ISO numeric codes match ACLED's `iso` field and avoid fragile country-name filters.
PROJECT_COUNTRIES = [
    {"iso": "120", "iso3": "CMR", "name": "Cameroon", "bloc_seed": "CEMAC"},
    {"iso": "140", "iso3": "CAF", "name": "Central African Republic", "bloc_seed": "CEMAC"},
    {"iso": "148", "iso3": "TCD", "name": "Chad", "bloc_seed": "CEMAC"},
    {"iso": "178", "iso3": "COG", "name": "Congo, Rep.", "bloc_seed": "CEMAC"},
    {"iso": "226", "iso3": "GNQ", "name": "Equatorial Guinea", "bloc_seed": "CEMAC"},
    {"iso": "266", "iso3": "GAB", "name": "Gabon", "bloc_seed": "CEMAC"},
    {"iso": "204", "iso3": "BEN", "name": "Benin", "bloc_seed": "ECOWAS"},
    {"iso": "854", "iso3": "BFA", "name": "Burkina Faso", "bloc_seed": "ECOWAS"},
    {"iso": "132", "iso3": "CPV", "name": "Cabo Verde", "bloc_seed": "ECOWAS"},
    {"iso": "384", "iso3": "CIV", "name": "Cote d'Ivoire", "bloc_seed": "ECOWAS"},
    {"iso": "270", "iso3": "GMB", "name": "Gambia", "bloc_seed": "ECOWAS"},
    {"iso": "288", "iso3": "GHA", "name": "Ghana", "bloc_seed": "ECOWAS"},
    {"iso": "324", "iso3": "GIN", "name": "Guinea", "bloc_seed": "ECOWAS"},
    {"iso": "624", "iso3": "GNB", "name": "Guinea-Bissau", "bloc_seed": "ECOWAS"},
    {"iso": "430", "iso3": "LBR", "name": "Liberia", "bloc_seed": "ECOWAS"},
    {"iso": "466", "iso3": "MLI", "name": "Mali", "bloc_seed": "ECOWAS"},
    {"iso": "562", "iso3": "NER", "name": "Niger", "bloc_seed": "ECOWAS"},
    {"iso": "566", "iso3": "NGA", "name": "Nigeria", "bloc_seed": "ECOWAS"},
    {"iso": "686", "iso3": "SEN", "name": "Senegal", "bloc_seed": "ECOWAS"},
    {"iso": "694", "iso3": "SLE", "name": "Sierra Leone", "bloc_seed": "ECOWAS"},
    {"iso": "768", "iso3": "TGO", "name": "Togo", "bloc_seed": "ECOWAS"},
]

FIELDS = [
    "event_id_cnty", "event_date", "year", "time_precision",
    "disorder_type", "event_type", "sub_event_type",
    "actor1", "assoc_actor_1", "inter1",
    "actor2", "assoc_actor_2", "inter2", "interaction",
    "civilian_targeting", "iso", "region", "country",
    "admin1", "admin2", "admin3", "location",
    "latitude", "longitude", "geo_precision",
    "source", "source_scale", "fatalities", "tags", "timestamp",
]

print("Catalog: cemac_ecowas_aes_trade")
print("Schema: bronze")
print(f"Source: ACLED API ({ACLED_READ_URL})")
print(f"Countries: {len(PROJECT_COUNTRIES)}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Limit per page: {LIMIT}")

In [ ]:
# Cell 2 - OAuth and request helpers
def get_secret(scope: str, key: str) -> str:
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception as exc:
        raise RuntimeError(
            f"Missing Databricks secret {scope}/{key}. Create the secret before running this notebook."
        ) from exc


def get_access_token() -> str:
    username = get_secret(SECRET_SCOPE, USERNAME_SECRET_KEY)
    password = get_secret(SECRET_SCOPE, PASSWORD_SECRET_KEY)
    response = requests.post(
        TOKEN_URL,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "username": username,
            "password": password,
            "grant_type": "password",
            "client_id": "acled",
            "scope": "authenticated",
        },
        timeout=60,
    )
    if response.status_code != 200:
        raise RuntimeError(
            f"ACLED OAuth failed with HTTP {response.status_code}. Check the Databricks secrets, not the notebook code."
        )
    payload = response.json()
    return payload["access_token"]


def acled_get(params: Dict[str, str], access_token: str) -> Dict:
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "User-Agent": "cemac-ecowas-aes-trade-observatory/0.1",
    }

    for attempt in range(1, MAX_RETRIES + 1):
        response = requests.get(
            ACLED_READ_URL,
            params=params,
            headers=headers,
            timeout=180,
        )

        if response.status_code == 401 and attempt == 1:
            access_token = get_access_token()
            headers["Authorization"] = f"Bearer {access_token}"
            continue

        if response.status_code in (429, 500, 502, 503, 504) and attempt < MAX_RETRIES:
            wait = RETRY_DELAY_SECONDS * attempt
            print(f"    HTTP {response.status_code}; retrying in {wait}s")
            time.sleep(wait)
            continue

        response.raise_for_status()
        payload = response.json()
        if int(payload.get("status", 200)) != 200:
            raise RuntimeError(f"ACLED returned non-200 payload status: {payload.get('status')}")
        return payload

    raise RuntimeError(f"ACLED request failed after {MAX_RETRIES} attempts")


def fetch_acled_country_year(country: Dict[str, str], year: int, access_token: str) -> List[Dict]:
    rows: List[Dict] = []
    page = 1
    while True:
        params = {
            "_format": "json",
            "iso": country["iso"],
            "year": str(year),
            "fields": "|".join(FIELDS),
            "limit": str(LIMIT),
            "page": str(page),
        }
        payload = acled_get(params, access_token)
        data = payload.get("data", []) or []
        rows.extend(data)

        print(f"    {country['iso3']} {year} page {page}: {len(data):,} rows")

        if len(data) < LIMIT:
            break
        if page >= MAX_PAGES_PER_COUNTRY_YEAR:
            raise RuntimeError(
                f"Reached page guard for {country['iso3']} {year}. Increase MAX_PAGES_PER_COUNTRY_YEAR."
            )

        page += 1
        time.sleep(SLEEP_BETWEEN_CALLS_SECONDS)

    return rows


def to_int(value):
    return int(value) if value not in (None, "") else None


def to_float(value):
    return float(value) if value not in (None, "") else None


print("Helper functions defined.")

In [ ]:
# Cell 3 - Authenticate once per notebook run
access_token = get_access_token()
print("ACLED OAuth token acquired.")

In [ ]:
# Cell 4 - Extract events by country and year
start_year = int(START_DATE[:4])
end_year = int(END_DATE[:4])
loaded_at = datetime.now(timezone.utc)

country_by_iso = {country["iso"]: country for country in PROJECT_COUNTRIES}
all_events = []
extraction_log = []
extraction_start = datetime.now(timezone.utc)

for country in PROJECT_COUNTRIES:
    print(f"Reporter {country['iso3']} ({country['name']}):")
    country_total = 0

    for year in range(start_year, end_year + 1):
        rows = fetch_acled_country_year(country, year, access_token)
        for row in rows:
            row["project_iso3"] = country["iso3"]
            row["project_country_name"] = country["name"]
            row["bloc_seed"] = country["bloc_seed"]
        all_events.extend(rows)
        country_total += len(rows)
        extraction_log.append({
            "iso3": country["iso3"],
            "year": year,
            "rows": len(rows),
        })
        time.sleep(SLEEP_BETWEEN_CALLS_SECONDS)

    print(f"  Total {country['iso3']}: {country_total:,} rows\n")

elapsed = (datetime.now(timezone.utc) - extraction_start).total_seconds()
print("=== EXTRACTION COMPLETE ===")
print(f"Events fetched: {len(all_events):,}")
print(f"Elapsed seconds: {elapsed:.1f}")

In [ ]:
# Cell 5 - Build event-level Spark DataFrame
if not all_events:
    raise ValueError("No ACLED events were fetched. Check credentials, date range, and country filters.")

event_rows = []
for event in all_events:
    event_rows.append(Row(
        event_id_cnty=event.get("event_id_cnty"),
        event_date=event.get("event_date"),
        year=to_int(event.get("year")),
        time_precision=to_int(event.get("time_precision")),
        disorder_type=event.get("disorder_type"),
        event_type=event.get("event_type"),
        sub_event_type=event.get("sub_event_type"),
        actor1=event.get("actor1"),
        assoc_actor_1=event.get("assoc_actor_1"),
        inter1=to_int(event.get("inter1")),
        actor2=event.get("actor2"),
        assoc_actor_2=event.get("assoc_actor_2"),
        inter2=to_int(event.get("inter2")),
        interaction=to_int(event.get("interaction")),
        civilian_targeting=event.get("civilian_targeting"),
        iso=event.get("iso"),
        iso3=event.get("project_iso3"),
        country=event.get("country"),
        project_country_name=event.get("project_country_name"),
        bloc_seed=event.get("bloc_seed"),
        region=event.get("region"),
        admin1=event.get("admin1"),
        admin2=event.get("admin2"),
        admin3=event.get("admin3"),
        location=event.get("location"),
        latitude=to_float(event.get("latitude")),
        longitude=to_float(event.get("longitude")),
        geo_precision=to_int(event.get("geo_precision")),
        source_name=event.get("source"),
        source_scale=event.get("source_scale"),
        fatalities=to_int(event.get("fatalities")),
        tags=event.get("tags"),
        acled_timestamp=to_int(event.get("timestamp")),
        source="acled",
        loaded_at=loaded_at,
    ))

events_df_raw = spark.createDataFrame(event_rows)
events_df = (
    events_df_raw
    .dropDuplicates(["event_id_cnty"])
    .withColumn("event_date_dt", F.to_date("event_date"))
)

print(f"Raw event rows: {events_df_raw.count():,}")
print(f"Deduped event rows: {events_df.count():,}")
events_df.printSchema()
events_df.show(10, truncate=False)

In [ ]:
# Cell 6 - Build weekly aggregate DataFrame
weekly_df = (
    events_df
    .withColumn(
        "week_start_date",
        F.expr("date_sub(event_date_dt, pmod(dayofweek(event_date_dt) + 5, 7))")
    )
    .groupBy(
        "iso3", "country", "project_country_name", "bloc_seed",
        "year", "week_start_date", "admin1", "event_type", "sub_event_type",
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum(F.coalesce(F.col("fatalities"), F.lit(0))).alias("fatalities"),
        F.countDistinct("event_id_cnty").alias("distinct_events"),
    )
    .withColumn("source", F.lit("acled"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

print(f"Weekly aggregate rows: {weekly_df.count():,}")
weekly_df.show(10, truncate=False)

In [ ]:
# Cell 7 - Write bronze tables
spark.sql("DROP TABLE IF EXISTS bronze.acled_events_historical")
spark.sql("DROP TABLE IF EXISTS bronze.acled_weekly_aggregated")

(events_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.acled_events_historical"))

(weekly_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.acled_weekly_aggregated"))

print("Write complete:")
print("  bronze.acled_events_historical")
print("  bronze.acled_weekly_aggregated")

In [ ]:
# Cell 8 - Verification and sanity checks
print("Event-level coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_events,
        COUNT(DISTINCT iso3) AS countries,
        MIN(event_date_dt) AS earliest_event_date,
        MAX(event_date_dt) AS latest_event_date,
        SUM(fatalities) AS total_fatalities
    FROM bronze.acled_events_historical
""").show()

print("Per-country event coverage:")
spark.sql("""
    SELECT
        iso3,
        project_country_name,
        COUNT(*) AS events,
        COUNT(DISTINCT year) AS years_present,
        SUM(fatalities) AS fatalities
    FROM bronze.acled_events_historical
    GROUP BY iso3, project_country_name
    ORDER BY events DESC
""").show(25, truncate=False)

print("Weekly aggregate coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS weekly_rows,
        COUNT(DISTINCT iso3) AS countries,
        MIN(week_start_date) AS earliest_week,
        MAX(week_start_date) AS latest_week,
        SUM(event_count) AS total_events_reconstructed,
        SUM(fatalities) AS total_fatalities_reconstructed
    FROM bronze.acled_weekly_aggregated
""").show()

print("Top event types since 2010:")
spark.sql("""
    SELECT
        event_type,
        COUNT(*) AS events,
        SUM(fatalities) AS fatalities
    FROM bronze.acled_events_historical
    GROUP BY event_type
    ORDER BY events DESC
""").show(truncate=False)

print("Cameroon admin1 conflict intensity, latest 3 years:")
spark.sql("""
    SELECT
        admin1,
        COUNT(*) AS events,
        SUM(fatalities) AS fatalities
    FROM bronze.acled_events_historical
    WHERE iso3 = 'CMR'
      AND year >= YEAR(CURRENT_DATE()) - 3
    GROUP BY admin1
    ORDER BY events DESC
    LIMIT 10
""").show(truncate=False)